# Week 5 Exercise — Personal Information Processor with RAG

A complete RAG pipeline rebuilt with **Anthropic Claude API** 

### Features
- Document loading from multiple folders (Markdown files)
- Intelligent text chunking with overlap
- Vector embeddings via Claude
- ChromaDB vector store for efficient retrieval
- t-SNE visualization (2D and 3D)
- Conversational RAG with memory
- Gradio chat interface
- Source attribution in answers

## 1. Install Dependencies

In [5]:
import sys
!{sys.executable} -m pip install -q gradio anthropic chromadb python-dotenv numpy plotly scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: c:\Users\Lenovo\projects\llm_engineering\.venv\Scripts\python.exe -m pip install --upgrade pip


## 2. Setup and Configuration

In [ ]:
import os
import re
import uuid
import json
import glob
import textwrap
from pathlib import Path
from typing import Optional

import numpy as np
import plotly.graph_objects as go
from sklearn.manifold import TSNE
import gradio as gr
import anthropic
import chromadb
from dotenv import load_dotenv

load_dotenv(override=True)


ANTHROPIC_MODEL    = "claude-sonnet-4-20250514"
DB_NAME            = "personal_knowledge_db"
CHUNK_SIZE         = 500     # characters per chunk
CHUNK_OVERLAP      = 100     # character overlap between chunks
TOP_K_RESULTS      = 5       # chunks to retrieve per query
KNOWLEDGE_BASE_PATH = Path("knowledge_base")

api_key = os.getenv("ANTHROPIC_API_KEY", "")
if api_key:
    print(f"Anthropic API Key found: {api_key[:15]}...")
else:
    print("⚠️  ANTHROPIC_API_KEY not set — add it to a .env file or set os.environ directly")

print("✅ Configuration ready")

c:\Users\Lenovo\projects\llm_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Anthropic API Key found: sk-ant-api03-me...
✅ Configuration ready


## 3. Global State

In [ ]:
_client:     Optional[anthropic.Anthropic]   = None
_chroma:     Optional[chromadb.Client]       = None
_collection: Optional[chromadb.Collection]  = None
_chat_history: list[dict] = []   


def get_client() -> anthropic.Anthropic:
    global _client
    if _client is None:
        key = os.getenv("ANTHROPIC_API_KEY", "")
        if not key:
            raise ValueError("ANTHROPIC_API_KEY not set")
        _client = anthropic.Anthropic(api_key=key)
    return _client


print("✅ Global state initialised")

✅ Global state initialised


## 4. Sample Knowledge Base

In [37]:
def create_sample_knowledge_base():
    """Create sample knowledge base with personal, projects, and learning data."""

    for folder in ["personal", "projects", "learning"]:
        (KNOWLEDGE_BASE_PATH / folder).mkdir(parents=True, exist_ok=True)

    files = {
        "personal/profile.md": """
# Personal Profile

## About Me
Name: Alex Johnson
Role: Software Engineer & AI Enthusiast
Location: Tech Hub City

## Background
I am a passionate software engineer with over 5 years of experience building scalable applications.
My journey started with web development and has evolved into specialising in AI and machine learning.
I completed my Computer Science degree at State University and have since worked at two startups.

## Skills
- Programming Languages: Python, JavaScript, TypeScript, Go, Rust
- Frameworks: React, FastAPI, LangChain, Gradio
- AI/ML: LLMs, RAG systems, Vector Databases, Prompt Engineering, Fine-tuning
- Databases: PostgreSQL, MongoDB, Chroma, Pinecone, Redis
- Cloud: AWS, GCP, Docker, Kubernetes

## Interests
I love exploring new technologies, contributing to open-source projects, and mentoring aspiring developers.
In my free time I enjoy hiking, reading tech blogs, and experimenting with new AI tools.
I run a small tech blog with 2,000 monthly readers.

## Contact
GitHub: github.com/alexj
LinkedIn: linkedin.com/in/alexjohnson
Email: alex@techmail.com
""",

        "projects/portfolio.md": """
# Projects Portfolio

## AI-Powered Document Assistant
A RAG-based system that helps users query large document collections efficiently.
Tech Stack: Python, LangChain, Chroma, Anthropic Claude API
Status: Shipped — used by 300+ beta users
Key Features: Semantic search, multi-document support, conversation history, source attribution
Lessons Learned: Chunking strategy matters enormously; smaller overlapping chunks outperform large ones.

## Real-time Analytics Dashboard
Built a scalable dashboard for visualising business metrics in real-time.
Tech Stack: React, Node.js, PostgreSQL, Redis, WebSockets
Impact: Reduced reporting time by 80% for the operations team.
Challenges: Handling 10,000 concurrent WebSocket connections required careful connection pooling.

## Code Review Automation Tool
An AI assistant that provides automated code reviews and suggestions.
Tech Stack: Python, GitHub API, Claude API
Features: Pattern detection, best practices recommendations, security vulnerability scanning
Status: Open-sourced on GitHub with 450 stars

## Personal Finance Tracker
Full-stack application for tracking expenses and predicting future spending patterns.
Tech Stack: React Native, FastAPI, PostgreSQL, ML forecasting
Features: Receipt scanning via OCR, budget alerts, spending trend analysis

## E-commerce Recommendation Engine
Built a collaborative filtering engine for a mid-size online retailer.
Tech Stack: Python, PyTorch, FastAPI, Redis
Impact: 23% increase in average order value after deployment.
""",

        "learning/journey.md": """
# Learning Journey

## Currently Learning (2025)
- Advanced RAG techniques: reranking, hybrid search, query decomposition
- Anthropic's Claude API: tool use, streaming, multi-turn conversations
- Rust programming language: ownership model, async Tokio runtime
- System design: distributed systems, consensus algorithms

## Completed Courses
- Deep Learning Specialisation — Coursera (Andrew Ng) — 2024
- Fullstack Open — University of Helsinki — 2022
- AWS Solutions Architect Associate — 2023
- Fast.ai Practical Deep Learning — 2024

## Books Read
- "Designing Data-Intensive Applications" — Martin Kleppmann
- "The Pragmatic Programmer" — Hunt & Thomas
- "Building Machine Learning Powered Applications" — Emmanuel Ameisen
- "Attention Is All You Need" — Vaswani et al. (paper)

## Certifications
- AWS Certified Solutions Architect — Associate (2023)
- Google Professional Data Engineer (2024)
- Certified Kubernetes Application Developer — CKAD (2023)

## Learning Goals for 2026
- Contribute to a major open-source AI project
- Build and ship a SaaS product from scratch
- Complete a Rust systems project
- Publish 3 technical blog posts on RAG architecture

## Study Schedule
Mornings (6-7am): Reading / papers
Lunch (12-1pm): Coding exercises
Evenings (8-9pm): Project work or courses
"""
    }

    for path, content in files.items():
        full_path = KNOWLEDGE_BASE_PATH / path
        if not full_path.exists():
            full_path.write_text(content.strip(), encoding="utf-8")
            print(f"  Created: {full_path}")
        else:
            print(f"  Exists:  {full_path}")

    print(f"\n✅ Knowledge base ready at: {KNOWLEDGE_BASE_PATH.resolve()}")


create_sample_knowledge_base()

  Exists:  knowledge_base\personal\profile.md
  Exists:  knowledge_base\projects\portfolio.md
  Exists:  knowledge_base\learning\journey.md

✅ Knowledge base ready at: C:\Users\Lenovo\Downloads\knowledge_base


## 5. Document Loading & Chunking

In [10]:
def load_documents() -> list[dict]:
    """
    Walk each subfolder of the knowledge base and load all .md files.
    Returns list of {text, doc_type, source}.
    """
    documents = []
    for folder in sorted(KNOWLEDGE_BASE_PATH.iterdir()):
        if not folder.is_dir():
            continue
        doc_type = folder.name
        for file in sorted(folder.rglob("*.md")):
            text = file.read_text(encoding="utf-8", errors="ignore")
            documents.append({
                "text":     text,
                "doc_type": doc_type,
                "source":   file.name,
            })
    print(f"Loaded {len(documents)} documents")
    print(f"Document types: {sorted(set(d['doc_type'] for d in documents))}")
    return documents


def chunk_document(doc: dict,
                   chunk_size: int = CHUNK_SIZE,
                   overlap: int = CHUNK_OVERLAP) -> list[dict]:
    """
    Split a document into overlapping character-level chunks.
    Each chunk inherits the document's metadata.
    """
    text   = doc["text"]
    chunks = []
    start  = 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append({
            "content":  text[start:end],
            "doc_type": doc["doc_type"],
            "source":   doc["source"],
        })
        if end == len(text):
            break
        start += chunk_size - overlap
    return chunks


# Load and chunk
documents = load_documents()

all_chunks: list[dict] = []
for doc in documents:
    all_chunks.extend(chunk_document(doc))

print(f"\nTotal chunks:        {len(all_chunks)}")
avg = sum(len(c['content']) for c in all_chunks) / len(all_chunks)
print(f"Average chunk size:  {avg:.0f} characters")
print(f"Chunks per doc type: { {t: sum(1 for c in all_chunks if c['doc_type']==t) for t in set(c['doc_type'] for c in all_chunks)} }")

Loaded 3 documents
Document types: ['learning', 'personal', 'projects']

Total chunks:        10
Average chunk size:  460 characters
Chunks per doc type: {'projects': 4, 'learning': 3, 'personal': 3}


## 6. Embeddings

Uses Claude to generate 128-dim semantic embeddings with robust JSON parsing and auto-retry.

In [ ]:
def _parse_embedding(raw: str) -> list[float]:
    """Robustly extract a float array from Claude's response."""
    raw = re.sub(r"```[a-z]*", "", raw).strip().strip("`").strip()

    try:
        vec = json.loads(raw)
        if isinstance(vec, list):
            return [float(x) for x in vec]
    except Exception:
        pass

    m = re.search(r"(\[.*?\])", raw, re.DOTALL)
    if m:
        candidate = re.sub(r",\s*]", "]", m.group(1))
        try:
            vec = json.loads(candidate)
            if isinstance(vec, list):
                return [float(x) for x in vec]
        except Exception:
            pass

    nums = re.findall(r"-?\d+\.\d+(?:[eE][+-]?\d+)?|-?\d+", raw)
    if nums:
        return [float(n) for n in nums]

    raise ValueError(f"Cannot parse embedding from:\n{raw[:300]}")


def embed_texts(texts: list[str], max_retries: int = 3) -> list[list[float]]:
    """
    Generate 128-dim semantic embeddings via Claude.
    Includes robust JSON parsing and retry logic.
    Swap for voyage-3 / text-embedding-3-small in production.
    """
    client = get_client()
    embeddings = []

    for i, text in enumerate(texts):
        safe = re.sub(r"[^\x20-\x7E]", " ", text).strip()[:800]
        prompt = (
            "Return a JSON array of exactly 128 floats (values between -1 and 1) "
            "representing the semantic embedding of the text below.\n"
            "Rules: output ONLY the JSON array starting with [ and ending with ]. "
            "No prose, no markdown. Use at most 6 decimal places. Do NOT truncate.\n\n"
            f"TEXT:\n{safe}"
        )
        vec = None
        for attempt in range(max_retries):
            try:
                resp = client.messages.create(
                    model=ANTHROPIC_MODEL,
                    max_tokens=1200,
                    messages=[{"role": "user", "content": prompt}]
                )
                vec = _parse_embedding(resp.content[0].text.strip())
                if len(vec) < 128:
                    vec += [0.0] * (128 - len(vec))
                vec = vec[:128]
                break
            except Exception as e:
                print(f"  [embed {i}] attempt {attempt+1} failed: {e}")

        if vec is None:
            raise RuntimeError(f"Embedding failed for chunk {i} after {max_retries} attempts")

        norm = sum(x**2 for x in vec) ** 0.5 or 1.0
        embeddings.append([x / norm for x in vec])

    return embeddings


print("✅ Embedding functions defined")

✅ Embedding functions defined


## 7. Build the Vector Store

In [ ]:
print(f"Embedding {len(all_chunks)} chunks (this may take a minute)...")

_chroma = chromadb.Client()   
col_name = "personal_rag"
if col_name in [c.name for c in _chroma.list_collections()]:
    _chroma.delete_collection(col_name)
_collection = _chroma.get_or_create_collection(
    col_name, metadata={"hnsw:space": "cosine"}
)

vectors = embed_texts([c["content"] for c in all_chunks])

_collection.add(
    ids       = [str(uuid.uuid4()) for _ in all_chunks],
    embeddings= vectors,
    documents = [c["content"]  for c in all_chunks],
    metadatas = [{"doc_type": c["doc_type"], "source": c["source"]} for c in all_chunks],
)

print(f"\n✅ Vector store ready — {_collection.count():,} chunks stored")
print(f"   Embedding dimensions: 128")

Embedding 10 chunks (this may take a minute)...

✅ Vector store ready — 10 chunks stored
   Embedding dimensions: 128


## 8. t-SNE Visualisation (2D & 3D)

In [ ]:

result     = _collection.get(include=["embeddings", "documents", "metadatas"])
vectors_np = np.array(result["embeddings"])
doc_types  = [m["doc_type"] for m in result["metadatas"]]
docs_text  = result["documents"]

COLOR_MAP = {"personal": "#7c6af7", "projects": "#f06292", "learning": "#4dd0e1"}
colors    = [COLOR_MAP.get(t, "#aaa") for t in doc_types]

n = vectors_np.shape[0]
perplexity = max(5.0, min(30.0, (n - 1) / 3.0))
print(f"Running t-SNE on {n} vectors (perplexity={perplexity:.1f})...")


tsne_2d = TSNE(n_components=2, random_state=42, perplexity=perplexity, max_iter=1000)
rv_2d   = tsne_2d.fit_transform(vectors_np)

fig_2d = go.Figure(data=[go.Scatter(
    x=rv_2d[:, 0], y=rv_2d[:, 1],
    mode="markers",
    marker=dict(size=7, color=colors, opacity=0.85,
                line=dict(width=0.5, color="white")),
    text=[f"Type: {t}<br>{d[:150]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])
fig_2d.update_layout(
    title="2D Vector Store Visualisation (t-SNE)",
    xaxis_title="t-SNE Dimension 1",
    yaxis_title="t-SNE Dimension 2",
    width=900, height=600,
    template="plotly_dark",
    paper_bgcolor="#0f0f17",
    plot_bgcolor="#1a1a28",
)
fig_2d.show()


tsne_3d = TSNE(n_components=3, random_state=42, perplexity=perplexity, max_iter=1000)
rv_3d   = tsne_3d.fit_transform(vectors_np)

fig_3d = go.Figure(data=[go.Scatter3d(
    x=rv_3d[:, 0], y=rv_3d[:, 1], z=rv_3d[:, 2],
    mode="markers",
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>{d[:150]}..." for t, d in zip(doc_types, docs_text)],
    hoverinfo="text"
)])
fig_3d.update_layout(
    title="3D Vector Store Visualisation (t-SNE)",
    scene=dict(
        xaxis_title="Dim 1", yaxis_title="Dim 2", zaxis_title="Dim 3",
        bgcolor="#1a1a28"
    ),
    width=1000, height=750,
    paper_bgcolor="#0f0f17",
)
fig_3d.show()

print("✅ Visualisations complete")

Running t-SNE on 10 vectors (perplexity=5.0)...


✅ Visualisations complete


## 9. RAG Chain

Query rewriting + retrieval + reranking + generation — all via Claude.

In [12]:
pip install nbformat

  Using cached nbformat-5.10.4-py3-none-any.whl.metadata (3.6 kB)
  Using cached fastjsonschema-2.21.2-py3-none-any.whl.metadata (2.3 kB)
Using cached nbformat-5.10.4-py3-none-any.whl (78 kB)
Using cached fastjsonschema-2.21.2-py3-none-any.whl (24 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: c:\Users\Lenovo\projects\llm_engineering\.venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""
    You are a helpful personal knowledge assistant.
    Answer questions accurately and concisely using ONLY the context provided.
    If the answer is not in the context, say so honestly.
    Always cite which source document your answer comes from.

    CONTEXT:
    {context}
""")


def retrieve_chunks(query: str, k: int = TOP_K_RESULTS) -> list[dict]:
    """Embed the query and fetch the top-k chunks from ChromaDB."""
    q_vec   = embed_texts([query])[0]
    results = _collection.query(
        query_embeddings=[q_vec],
        n_results=min(k, _collection.count())
    )
    return [
        {"content": doc, "doc_type": meta["doc_type"], "source": meta["source"]}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]


def chat(question: str, history: list) -> str:
    """
    Full RAG pipeline:
      1. Retrieve relevant chunks
      2. Build context
      3. Generate answer with Claude (using full conversation history)
    """
    global _chat_history

    if _collection is None:
        return "⚠️ Vector store not built yet — run the cells above first."

    try:
        client = get_client()

      
        chunks = retrieve_chunks(question)

        
        context = "\n\n---\n\n".join(
            f"[{c['doc_type']} / {c['source']}]\n{c['content']}"
            for c in chunks
        )
        sources = sorted(set(c["doc_type"] for c in chunks))

    
        response = client.messages.create(
            model=ANTHROPIC_MODEL,
            max_tokens=1024,
            system=SYSTEM_PROMPT.format(context=context),
            messages=_chat_history + [{"role": "user", "content": question}]
        )
        answer = response.content[0].text.strip()

    
        _chat_history.append({"role": "user",      "content": question})
        _chat_history.append({"role": "assistant",  "content": answer})

        return answer + f"\n\n_Sources: {', '.join(sources)}_"

    except Exception as e:
        return f"❌ Error: {e}"


print("RAG chain ready. Running smoke test...\n")
_chat_history = []
test_q = "What is the person's background?"
print(f"Q: {test_q}")
print(f"A: {chat(test_q, [])}")

RAG chain ready. Running smoke test...

Q: What is the person's background?
A: Based on the profile document, here's the person's background:

**Professional Background:**
- Currently works as a Senior Software Engineer at TechCorp
- Previously worked at two startups before joining TechCorp

**Technical Skills:**
- Programming Languages: Python, JavaScript, TypeScript, Go, Rust
- Frameworks: React, FastAPI, LangChain, Gradio
- AI/ML: LLMs, RAG systems, Vector Databases, Prompt Engineering, Fine-tuning
- Databases: PostgreSQL, MongoDB, Chroma, Pinecone, Redis
- Cloud: AWS, GCP, Docker, Kubernetes

**Certifications:**
- AWS Certified Solutions Architect — Associate (2023)
- Google Professional Data Engineer (2024)
- Certified Kubernetes Application Developer — CKAD (2023)

**Personal Interests:**
- Exploring new technologies
- Contributing to open-source projects
- Mentoring aspiring developers
- Hiking and reading tech blogs
- Experimenting with new AI tools
- Runs a small tech blog wit

## 10. Gradio Chat Interface

In [ ]:
def create_gradio_interface():
    """Build and return the Gradio Blocks UI."""

    THEME = gr.themes.Base(
        primary_hue="violet",
        secondary_hue="purple",
        neutral_hue="slate",
        font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "sans-serif"],
    ).set(
        body_background_fill="#0f0f17",
        body_text_color="#e2e0f0",
        block_background_fill="#1a1a28",
        block_border_color="#2d2d45",
        block_title_text_color="#c4b5fd",
        input_background_fill="#12121e",
        button_primary_background_fill="linear-gradient(135deg, #7c3aed, #a855f7)",
        button_primary_text_color="#fff",
    )

    with gr.Blocks(theme=THEME) as ui:

        gr.HTML("""
        <div style="text-align:center; padding:1.5rem 0 0.5rem;">
          <h1 style="font-size:2rem; font-weight:800;
                     background:linear-gradient(135deg,#a78bfa,#f472b6);
                     -webkit-background-clip:text; -webkit-text-fill-color:transparent;
                     margin-bottom:6px;">
            🧠 Personal Knowledge Worker
          </h1>
          <p style="color:#8b8aaa; font-size:0.9rem;">
            Powered by Claude · RAG over your personal knowledge base
          </p>
        </div>
        """)

        with gr.Tabs():

            with gr.Tab("💬 Chat"):
                gr.ChatInterface(
                    fn=chat,
                    title="",
                    description="Ask anything about your personal data",
                    examples=[
                        "What is my background?",
                        "Tell me about my projects",
                        "What am I currently learning?",
                        "What are my main skills?",
                        "What certifications do I have?",
                    ],
                )

        
            with gr.Tab("📁 Knowledge Base"):
                gr.Markdown("### Files in the knowledge base")
                kb_info = gr.Textbox(
                    value="\n".join(
                        f"[{c['doc_type']}] {c['source']}  ({len(c['content'])} chars)"
                        for c in all_chunks[:20]
                    ) + (f"\n…and {len(all_chunks)-20} more chunks" if len(all_chunks) > 20 else ""),
                    lines=15, interactive=False,
                    label="Chunks preview"
                )
                gr.Markdown(
                    f"**Total chunks:** {len(all_chunks)}  ·  "
                    f"**Vector dimensions:** 128  ·  "
                    f"**Model:** {ANTHROPIC_MODEL}"
                )

            with gr.Tab("ℹ️ About"):
                gr.Markdown("""
## RAG Pipeline

### How it works
1. **Load** — Markdown files are read from `knowledge_base/personal/`, `projects/`, `learning/`
2. **Chunk** — Documents are split into 500-character overlapping chunks
3. **Embed** — Each chunk is embedded into a 128-dim vector via Claude
4. **Store** — Vectors are stored in an in-memory ChromaDB collection
5. **Retrieve** — At query time, the question is embedded and top-5 chunks retrieved by cosine similarity
6. **Generate** — Claude answers using only the retrieved context, with full conversation history

### Tech stack
- **LLM & Embeddings**: Anthropic Claude (`claude-sonnet-4-20250514`)
- **Vector store**: ChromaDB (in-memory)
- **Visualisation**: t-SNE via scikit-learn + Plotly
- **UI**: Gradio

### Tips
- Ask specific questions for better answers
- Follow-up questions work thanks to conversation memory
- Add your own `.md` files to the `knowledge_base/` subfolders and re-run the notebook
                """)

        gr.HTML("""
        <div style="text-align:center;padding:1rem 0 0.5rem;color:#555;font-size:0.75rem;">
          Week 5 Exercise · Personal Knowledge Worker · Built with Gradio + Claude
        </div>
        """)

    return ui

print("✅ Gradio interface configured")


✅ Gradio interface configured


## 11. Launch the Application

In [ ]:
_chat_history = []   

ui = create_gradio_interface()

print("\nLaunching Personal Knowledge Worker...")
print("Open the URL shown below in your browser.")
print("Press the Stop button in the notebook toolbar to shut down.\n")

ui.launch(share=False, server_port=7869)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20300\2300027294.py:20: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=THEME) as ui:



Launching Personal Knowledge Worker...
Open the URL shown below in your browser.
Press the Stop button in the notebook toolbar to shut down.

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
